# Scaled Dot-Product Attention — Attention Is All You Need (2017)

_Papers · Transformers & LLMs_

**Let every token look at every other token: weight values by how well a query matches each key, with one scaled-dot-product softmax.**

---

This notebook is a practice scaffold for the **Scaled Dot-Product Attention — Attention Is All You Need (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn.functional as F

torch.manual_seed(0)

def my_attention(Q, K, V):
    """Scaled dot-product attention — Eq.(1) of Vaswani et al. (2017), Section 3.2.1.
       Q:(n_q,d_k)  K:(n_k,d_k)  V:(n_k,d_v)  ->  (n_q,d_v), plus the attention map."""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1)        # (n_q, n_k): query i vs key j
    scores = scores / (d_k ** 0.5)          # scale by 1/sqrt(d_k)
    weights = scores.softmax(dim=-1)        # softmax over KEYS (last dim); each row sums to 1
    out = weights @ V                       # weighted average of the values
    return out, weights

# ---- THE ORACLE: my version must equal F.scaled_dot_product_attention ----
n_q, n_k, d_k, d_v = 3, 5, 8, 16
Q = torch.randn(n_q, d_k); K = torch.randn(n_k, d_k); V = torch.randn(n_k, d_v)
mine, _ = my_attention(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V)   # PyTorch's built-in, same scaling
print("allclose vs F.scaled_dot_product_attention:", torch.allclose(mine, ref, atol=1e-6))  # expect True

# show that DROPPING the scale breaks the match (the ablation)
def unscaled(Q, K, V):
    return (Q @ K.transpose(-2,-1)).softmax(dim=-1) @ V
print("allclose if we forget 1/sqrt(d_k):", torch.allclose(unscaled(Q,K,V), ref, atol=1e-6))  # expect False

# ---- recompute the worked example: 2 tokens, d_k=d_v=2 ----
Qe = torch.eye(2); Ke = torch.eye(2); Ve = torch.tensor([[10.,0.],[0.,10.]])
out_e, w_e = my_attention(Qe, Ke, Ve)
print("worked-example attention map:\n", w_e)        # ~ [[0.6698,0.3302],[0.3302,0.6698]]
print("worked-example output:\n", out_e)             # ~ [[6.698,3.302],[3.302,6.698]]

# ---- a tiny attention map you can read ----
torch.manual_seed(1)
Qt = torch.randn(4, 8); Kt = torch.randn(4, 8); Vt = torch.randn(4, 8)
_, amap = my_attention(Qt, Kt, Vt)
print("4x4 attention map (rows=queries, cols=keys, each row sums to 1):")
for row in amap.tolist():
    print("  ", [round(x, 2) for x in row])
print("row sums:", [round(s, 3) for s in amap.sum(-1).tolist()])  # all ~1.0

## Visualize the data & results

_On a tiny set of tokens, what does the scaled-dot-product attention map look like — does each query concentrate weight on the key it matches best, and does each row sum to 1?_

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(1)

# 4 toy tokens, d_k = 8; same tensors are Q, K and V (self-attention style)
Q = torch.randn(4, 8); K = torch.randn(4, 8); V = torch.randn(4, 8)
d_k = Q.shape[-1]
amap = (Q @ K.transpose(-2,-1) / d_k**0.5).softmax(dim=-1)   # (4,4) attention map

print("full attention map (rows sum to 1):")
for r in amap.tolist():
    print("  ", [round(x,4) for x in r])
print("query 0 weights:", [round(x,4) for x in amap[0].tolist()])  # ~[0.0841,0.1228,0.0421,0.7510]
print("row sums:", [round(s,3) for s in amap.sum(-1).tolist()])    # all 1.0

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A query has scores $[2, 0]$ against two keys, with $d_k=4$. Compute the scaled scores and the attention weights.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scale by $\sqrt{d_k}=\sqrt4=2$: $[2/2,\,0/2]=[1,\,0]$. — _Equation (1) divides every score by $\sqrt{d_k}$._
- Softmax: $e^1\approx2.718$, $e^0=1$, sum $3.718$. Weights $[2.718/3.718,\,1/3.718]=[0.731,\,0.269]$. — _Turns scores into positive weights summing to 1._

**Answer:** Scaled scores $[1,0]$; attention weights $[0.731, 0.269]$. The query puts about 73% of its weight on key 1.

</details>

**Problem 2.** Ablation: remove the $1/\sqrt{d_k}$ scaling and use large keys/queries (say $d_k=64$, entries ~$N(0,1)$). What happens to the attention weights and the gradient?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the raw dot products now have standard deviation $\sqrt{d_k}=8$, so scores like $\pm16$ are common. — _Variance of a $d_k$-term dot product is $d_k$._
- Softmax of scores that differ by ~16 is nearly one-hot, e.g. $[0.9999,\dots]$. — _Large gaps saturate the softmax._
- At a near one-hot softmax, its Jacobian (sensitivity) is almost zero, so almost no gradient flows back. — _This is exactly the failure the scaling prevents._

**Answer:** Without scaling the weights collapse to nearly one-hot and the softmax's gradient nearly vanishes, so learning stalls — the paper's stated reason for the $1/\sqrt{d_k}$ factor (Section 3.2.1). The CODE cell shows the unscaled version no longer matches PyTorch.

</details>

**Problem 3.** If $Q$ is $(3, 8)$, $K$ is $(5, 8)$, and $V$ is $(5, 16)$, what is the shape of the attention map and of the output?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Score matrix is $QK^\top$: $(3,8)\times(8,5)=(3,5)$. — _Each of 3 queries scored against each of 5 keys._
- Softmax does not change shape, so the attention map is $(3,5)$. — _Weights, one row per query over the 5 keys._
- Output is (weights)$\,V$: $(3,5)\times(5,16)=(3,16)$. — _Each query returns a $d_v=16$-length blended value._

**Answer:** Attention map $(3,5)$; output $(3,16)$. Note $d_k=8$ must match between $Q$ and $K$, while $d_v=16$ sets the output width.

</details>